In [ ]:
!pip3 install kafka-python-ng
!pip install boto3


In [ ]:
from kafka import KafkaProducer
import json
import time

In [ ]:
import os
import boto3
from kafka import KafkaProducer
import json
from botocore.exceptions import NoCredentialsError, ClientError

# Optional: Set AWS credentials using environment variables
os.environ["AWS_ACCESS_KEY_ID"] = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""
os.environ["AWS_DEFAULT_REGION"] = "ap-south-1"

# S3 Setup
bucket_name = 'ashishbucket08'
file_key = 'zomato.csv'

try:
    # Connect to S3
    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket=bucket_name, Key=file_key)
    lines = obj['Body'].read().decode('utf-8').splitlines()

    # Skip header and take first 10 rows
    data_lines = lines[1:]

    # Kafka Setup
    producer = KafkaProducer(
        bootstrap_servers='3.110.108.178:9092',
        value_serializer=lambda v: json.dumps(v).encode('utf-8')
    )

    # Send each line to Kafka
    for line in data_lines:
        if line.strip():  # Skip empty lines
            data = {'data': line}
            producer.send('zomato-topic', value=data)
            #print(f"Sent to Kafka: {data}")

    producer.flush()
    producer.close()
    print("✅ First 10 data lines sent to Kafka topic.")

except NoCredentialsError:
    print("❌ AWS credentials not found. Set them using environment variables or AWS config.")
except ClientError as e:
    print(f"❌ Error accessing S3: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")
